# 判定の門の学習（コンパウンドIME 用）

「この変換で人工知能の生成（all_zenzai）が要るか」を決める、文字単位の小さな模型（約450万の数）を学習します。
入力は前の文・読み・生成なしの候補の上位、出力は「生成が要る確率」と候補ごとの点です。
今回（v4sel）は **答えを選ぶ頭** を足しました。生成した v5 の出力を「もう1つの候補の枠」として入れ、
(b) 候補4個と v5 の出力のどれが正しいか、(a) v5 の出力と生成なしの候補の一番のどちらが正しいか、を選びます
（v5 の出力の枠は他から見えないので、生成の前の門の出力は前と同じ形です。選ぶ頭は生成のあとにしか使えません。`docs/jev-gate.md` の「答えを選ぶ頭」）。
材料（学習用の文の記録）は GitHub Actions で作った約100万文の記録（`records-01.jsonl.gz`〜`records-10.jsonl.gz` と、試験の組の記録 `test-records.jsonl.gz`。
手元の `ContextIME/release/jevgate-material-1m`。01〜05 は前回の50万文、06〜10 が今回足した50万文）です（`docs/jev-gate.md`）。

**使い方**

1. 上のメニューの「ランタイム」→「ランタイムのタイプを変更」で **L4 GPU** を選んで「保存」を押します
   （T4 でも動きます。CPU でも動きますが遅いです）。
2. 下のセルを **上から順に1つずつ** 押します（セルの左の ▶ ボタン）。
3. 途中で接続が切れたら、もう一度 **①から順に** 押してください。終わった所は飛ばし、学習は最後の
   チェックポイント（10分ごとにドライブへ保存）から続きます。
4. 材料はドライブの `jev-gate/material/m2` にある11個のファイル（01〜10 と試験の組。v3 のときに上げたもの）がそのまま使えます。新しく上げるものはありません（⑤が件数を確かめます）。

かかる時間の目安（L4、100万文・4候補・最大4周）: 特徴づくりと読み込みに十数分〜30分、学習は1周 約26〜31分（10万文のときの速さ 毎秒約2.5歩 からの見込み。枠が4個から5個に増えるぶん数割遅くなる見込みで、未測定。1周 約3,800歩）。
確かめ用の AUC が一定回数よくならなければ早めに止めて一番良い結果を使うので、全体で約1.3〜2.7時間の見込みです（4周まで流れたとき約2.7時間。未測定）。⑦が始まると実際の速さで残り時間を出し直します。
**できた模型はまだ公開しないでください。** 試験の組の机上の評価（⑨）で、生成に戻すしきい値ごとの正解数を確かめてから決めます。

## ① 設定
ふつうは変えなくてかまいません。▶ を押してください。

In [ ]:
#@title ① 設定
RUN_NAME = "v4sel"          # 結果を置くフォルダの名前。別の設定で作り直すときは名前を変える
REPO_URL = "https://github.com/J-KAMMA/compound-ime.git"
REPO_REF = "exp/jev-gate"  # 使うリポジトリの枝または版
GH_REPO = "J-KAMMA/compound-ime"
MATERIAL_RUN = ""          # 材料を作った GitHub Actions の run の番号（⑤で自動で取るとき。空なら自分で置いた材料を使う）
MATERIAL_ARTIFACT = ""     # ⑤で自動で取る成果物の名前（今回の100万文は手で置くので空）
MATERIAL_NAME = "m2"       # 材料を置くフォルダの名前（ドライブの jev-gate/material/<名前>）。m2 には 01〜10（100万文）と試験の組が入っている

K = 4                      # 入力に入れる生成なしの候補の数（0〜16）。大きいほど正確だが推論が遅い（docs/jev-gate.md の時間の表）
D, LAYERS, HEADS, FFN = 256, 4, 4, 1024   # 模型の大きさ。d=256・4層で約450万、d=192・3層で約220万
EPOCHS = 4                 # 同じ記録を何回くり返すか（上限。100万文では 1周 約3,800歩。早めに止まることが多い）
EVAL_EVERY = 500           # 確かめ用の AUC を測る歩の間隔（周の終わりにも測る。測るたびに最良の結果を残す）
METRIC = "need_k+pair"    # 早めに止める基準。門 need_k と選ぶ頭 a の AUC の平均（選ぶ頭が育つまで止めないため）。"need_k" だけにも変えられる
PATIENCE = 5               # 確かめ用の AUC がこの回数つづけて良くならなければ止めて、一番良い結果を使う（覚えすぎを防ぐ）
BATCH = 256                # 1歩の文の数
LEARNING_RATE = 6e-4
SEED = 20261019
CHECKPOINT_MINUTES = 10    # 途中の結果をドライブに残す間隔（分）
LIMIT = None               # 試しに少なく読むときの文の数（None なら全部）

DRIVE_ROOT = "/content/drive/MyDrive/jev-gate"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
MATERIAL_DIR = f"{DRIVE_ROOT}/material/{MATERIAL_NAME}"
WORK = "/content/work"
print("設定しました。結果の置き場所:", RUN_DIR)

## ② GPU の確認と Google ドライブへの接続
▶ を押すと、Google ドライブへの接続の許可を求める画面が出ます。**自分のアカウントを選んで許可**してください。

In [ ]:
#@title ② GPU の確認と Google ドライブへの接続
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし（CPU。遅いです）")
from google.colab import drive
drive.mount("/content/drive")
for d in (RUN_DIR, MATERIAL_DIR, WORK):
    os.makedirs(d, exist_ok=True)
print("ドライブにつながりました:", RUN_DIR)

## ③ 台本を取ってくる
学習の台本（`tools/jev-gate/`）と評価の道具を GitHub から取ります。

In [ ]:
#@title ③ 台本を取ってくる
import os, subprocess, sys, importlib
REPO = f"{WORK}/compound-ime"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
KIT = f"{REPO}/tools/jev-gate"
sys.path.insert(0, KIT)
sys.path.insert(0, f"{REPO}/tools")
import jevgate as jg
importlib.reload(jg)
print("リポジトリの版:", REPO_COMMIT)
cfg = dict(k=K, d=D, layers=LAYERS, heads=HEADS, ffn=FFN)
m = jg.JevGate(4096, {**jg.DEFAULT_CFG, **cfg})
print(f"模型の数: 約{jg.count_params(m) / 1e6:.2f} 百万（語彙 4096 のとき）")
del m

## ④ 道具の確認
Colab には torch と numpy が入っているので、追加で入れるものはありません。

In [ ]:
#@title ④ 確認
import torch, numpy
print("torch", torch.__version__, " numpy", numpy.__version__)

## ⑤ 材料を取ってくる
学習用の記録 `records-01.jsonl.gz`〜`records-10.jsonl.gz` と、試験の組の記録 `test-records.jsonl.gz`（合わせて11個、約460MB）を
ドライブの `jev-gate/material/<MATERIAL_NAME>`（既定 `m2`）に置きます。

- **自分で置く（今回の材料はこちら）**: `m2` には前回（v3）で上げた 01〜10 と試験の組がそのまま入っているので、新しく上げるものはありません
  （ドライブの `マイドライブ/jev-gate/material/m2` に11個あることだけ確かめてください。下で件数を確かめます）。
- **自動で取る**: 材料を GitHub Actions の成果物としてまとめたときだけ使えます。①の `MATERIAL_RUN` に run の番号、
  `MATERIAL_ARTIFACT` に成果物の名前を入れ、Colab の左のカギのアイコン（シークレット）に
  `GH_TOKEN`（リポジトリ J-KAMMA/compound-ime の「Actions: 読み取り」だけの細かい権限のトークン）を登録して
  「ノートブックからのアクセス」をオンにします。

In [ ]:
#@title ⑤ 材料を取ってくる
import glob, io, json, os, urllib.request, zipfile
have = sorted(glob.glob(f"{MATERIAL_DIR}/records-*.jsonl.gz"))
if have and os.path.exists(f"{MATERIAL_DIR}/test-records.jsonl.gz"):
    print("材料はもうそろっています:", len(have), "ファイル")
elif MATERIAL_RUN:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
    def api(url):
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"})
        return urllib.request.urlopen(req)
    arts, page = [], 1
    while True:
        res = json.load(api(f"https://api.github.com/repos/{GH_REPO}/actions/runs/{MATERIAL_RUN}/artifacts?per_page=100&page={page}"))
        arts += res["artifacts"]
        if len(res["artifacts"]) < 100:
            break
        page += 1
    for a in arts:
        if a["name"] != MATERIAL_ARTIFACT or a["expired"]:
            continue
        data = api(a["archive_download_url"]).read()
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for n in z.namelist():
                if n.endswith(".jsonl.gz"):
                    open(f"{MATERIAL_DIR}/{os.path.basename(n)}", "wb").write(z.read(n))
        print("取りました:", a["name"], a["size_in_bytes"] // 1_000_000, "MB")
else:
    raise SystemExit(f"材料がありません。{MATERIAL_DIR} に records-*.jsonl.gz と test-records.jsonl.gz を置くか、①の MATERIAL_RUN を入れてください。")
RECORDS = sorted(glob.glob(f"{MATERIAL_DIR}/records-*.jsonl.gz"))
TEST_RECORDS = f"{MATERIAL_DIR}/test-records.jsonl.gz"
print("学習用の記録:", len(RECORDS), "ファイル。試験の組の記録:", os.path.exists(TEST_RECORDS))
if len(RECORDS) != 10:
    print("※ 記録のファイルが10個ではありません（01〜10 の100万文の想定）。足りなければ上げてからもう一度押してください。")

## ⑥ 件数とラベルの割合
学習用の文の数、「生成が要る」ラベルの割合などを出します（約100万文、生成が要る 24〜25% の見込み）。

In [ ]:
#@title ⑥ 件数とラベルの割合
import subprocess
print(subprocess.run([sys.executable, f"{REPO}/tools/jev_gate_build.py", "stats", *RECORDS], capture_output=True, text=True).stdout)

## ⑦ 学習する（いちばん長い）
**途中で切れても、①から押し直せば続きから始まります。** チェックポイントは10分ごとにドライブへ保存されます。

In [ ]:
#@title ⑦ 学習する
import os
FINAL = f"{RUN_DIR}/run/final.pt"
if os.path.exists(FINAL):
    print("学習はもう終わっています:", FINAL)
else:
    jg.train(RECORDS, f"{RUN_DIR}/run", cfg, epochs=EPOCHS, batch_size=BATCH, lr=LEARNING_RATE, seed=SEED,
             limit=LIMIT, checkpoint_minutes=CHECKPOINT_MINUTES, cache_dir=f"{WORK}/cache",
             eval_every=EVAL_EVERY, patience=PATIENCE, metric=METRIC)

## ⑧ 確かめ用の成績
学習に使っていない文（記録の id の指紋で2%）での、各頭の AUC と、生成に回す割合ごとの正解数です。

In [ ]:
#@title ⑧ 確かめ用の成績
import json
rows = [json.loads(l) for l in open(f"{RUN_DIR}/run/log.txt", encoding="utf-8") if l.startswith("{")]
best_step = rows[-1].get("best_step") if rows else None
print("確かめ用の AUC（need_k）の推移（歩: AUC）:", "、".join(f"{r['step']}: {r['eval']['auc']['need_k']:.3f}" for r in rows))
print("選ぶ頭 a（v5 と候補の一番）の AUC の推移:", "、".join(f"{r['step']}: {r['eval']['auc'].get('pair', float('nan')):.3f}" for r in rows))
print("いちばん良かった歩（最終の結果に使った）:", best_step)
for r in rows:
    if r["step"] == best_step:
        print("歩", r["step"], "（最良）")
        jg.print_eval(r["eval"])
print("最後の測定")
jg.print_eval(rows[-1]["eval"])
# 選ぶ頭は、割れる文（v5 と候補の一番の片方だけ正解）で「当て推量（AUC 0.5）」より良いか、(b) の正解数が「いつも v5」を上回るかを見る

## ⑨ 試験の組での机上の評価
`run 39249431055` の材料から作った試験の組の記録（学習に使っていない）に模型を当て、しきい値ごとに
「正解数」「生成に回る割合」「速さの見込み」を出します。
`student`（今の生成なしの選び方）、`zenz`（上限）に加えて、`model2`（答えを選ぶ頭 b）、`model2a`（頭 a）で生成のあとの答えを選んだときの表と、
同じしきい値で4つの選び方を並べた「選び方の比べ」の表を出します。**しきい値は⑧の確かめ用で決めてください。**
ここで試験の文を見て決めると本番より良く見えます。

In [ ]:
#@title ⑨ 試験の組での机上の評価
import subprocess, sys
PRED = f"{RUN_DIR}/result/pred-test.tsv"
os.makedirs(f"{RUN_DIR}/result", exist_ok=True)
jg.predict(f"{RUN_DIR}/run/final.pt", [TEST_RECORDS], PRED)
for select in ("student", "zenz", "model2", "model2a"):
    out = subprocess.run([sys.executable, f"{KIT}/gate_eval.py", TEST_RECORDS, "--pred", PRED, "--head", "need_k",
                          "--select", select, "--by-set"], capture_output=True, text=True)
    print(out.stdout)
    open(f"{RUN_DIR}/result/eval-{select}.md", "w", encoding="utf-8").write(out.stdout)

## ⑩ 書き出し
C++ の推論用の重みの表（JEVG 形式、単精度と半精度）と語彙、C++ との答え合わせ用の見本を `result` に置きます。

In [ ]:
#@title ⑩ 書き出し
jg.export(f"{RUN_DIR}/run/final.pt", f"{RUN_DIR}/result/jevgate-fp32.bin")
jg.export(f"{RUN_DIR}/run/final.pt", f"{RUN_DIR}/result/jevgate-fp16.bin", fp16=True)
jg.dump_case(f"{RUN_DIR}/run/final.pt", [TEST_RECORDS], f"{RUN_DIR}/result/case.txt")
import shutil
shutil.copy(f"{RUN_DIR}/run/log.txt", f"{RUN_DIR}/result/train-log.txt")
print("できあがり。Google ドライブの次の場所にあります:", f"{RUN_DIR}/result")
print("次の手順: 手元で tools/jev-gate/jevgate_infer.cpp を組み立て、`jevgate_infer check jevgate-fp32.bin case.txt` で答え合わせ、")
print("`jevgate_infer bench --model jevgate-fp16.bin --k <K> --threads 4` で時間を測ります（docs/jev-gate.md）。")
print("C++ の check は、v5 の出力の枠つきの見本と、枠を外した入力の門が同じことも確かめます。")
print("※ この模型はまだ公開しないでください。")